# Notebook 01: Preprocesamiento y Limpieza de Datos (Data Engineering)
**Proyecto:** Hit Predictor & Trendy Dashboard — Grupo 5   

---

### Objetivos de este Cuaderno:
1. **Consolidar Datasets:** Unificar las fuentes de canciones de alta popularidad (`high_popularity`) y baja popularidad (`low_popularity`).
2. **Depuración de Calidad:** Identificar y eliminar duplicados únicos por `track_id` y remover filas con valores nulos en variables críticas.
3. **Ingeniería de Características:**
   * Convertir la duración de canciones de milisegundos (`duration_ms`) a minutos (`duration_min`).
   * Definir la variable binaria de éxito comercial `is_hit` ($\ge 70$ popularidad).
4. **Exportación:** Generar el archivo maestro limpio `data/spotify_clean.csv` listo para la fase de Machine Learning.

In [74]:
import pandas as pd
import numpy as np
import os
import csv

# Configuración para visualizar todas las columnas en exploraciones
pd.set_option('display.max_columns', None)

print("Librerías cargadas exitosamente.")

Librerías cargadas exitosamente.


## 1. Carga de Datasets Originales
Cargamos los dos archivos CSV proporcionados. 
> Ambos archivos utilzan `,` como separador de columnas y la codificación `latin-1` debido a caracteres especiales en los nombres de canciones y artistas.

In [75]:
def reparar_encoding(texto):
    if not isinstance(texto, str):
        return texto
    try:
        # Convierte de latin-1 a UTF-8 real
        return texto.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return texto

# Rutas de los archivos
PATH_HIGH = "../data/high_popularity_spotify_data.csv"
PATH_LOW = "../data/low_popularity_spotify_data.csv"

# Lectura con codificación y separador adecuado
df_high = pd.read_csv(PATH_HIGH, sep=",", encoding="latin-1")
df_low = pd.read_csv(PATH_LOW, sep=",", encoding="latin-1")

print(f"Canciones de Alta Popularidad cargadas: {df_high.shape[0]} filas | {df_high.shape[1]} columnas")
print(f"Canciones de Baja Popularidad cargadas: {df_low.shape[0]} filas | {df_low.shape[1]} columnas")

Canciones de Alta Popularidad cargadas: 1686 filas | 29 columnas
Canciones de Baja Popularidad cargadas: 3145 filas | 29 columnas


## 2. Consolidación y Depuración de Duplicados
Unimos ambos registros en un solo DataFrame y procedemos a eliminar registros duplicados evaluando su ID único de Spotify (`track_id`).

In [76]:
cols_texto = ['track_name', 'track_artist', 'track_album_name', 'playlist_name']

for col in cols_texto:
    if col in df_high.columns:
        df_high[col] = df_high[col].apply(reparar_encoding)
    if col in df_low.columns:
        df_low[col] = df_low[col].apply(reparar_encoding)

# Concatenar DataFrames
df_combined = pd.concat([df_high, df_low], ignore_index=True)
registros_iniciales = len(df_combined)

# Eliminar duplicados basados en 'track_id'
df_clean = df_combined.drop_duplicates(subset=['track_id']).copy()
duplicados_removidos = registros_iniciales - len(df_clean)

print(f"Registros consolidados iniciales: {registros_iniciales}")
print(f"Filas duplicadas eliminadas: {duplicados_removidos}")
print(f"Total registros únicos restantes: {len(df_clean)}")

Registros consolidados iniciales: 4831
Filas duplicadas eliminadas: 336
Total registros únicos restantes: 4495


## 3. Ingeniería de Características (Transformaciones)
Para facilitar el análisis e interpretación:
1. **Transformación de Tiempo:** Convertimos `duration_ms` a minutos dividiendo por 60,000.
2. **Etiqueta Objetivo (`is_hit`):** 
   * `1` (**Hit Commercial**): Para canciones con `track_popularity >= 70`.
   * `0` (**Estándar / No-Hit**): Para canciones con `track_popularity < 70`.
3. Inspeccionamos y removemos registros con valores nulos en métricas clave.

In [77]:
# 1. Transformar duración a minutos
df_clean['duration_min'] = (df_clean['duration_ms'] / 60000).round(2)

# 2. Generar variable binaria 'is_hit'
df_clean['is_hit'] = (df_clean['track_popularity'] >= 70).astype(int)

# 3. Convertir cadenas vacías o espacios a NaN en 'track_id' para detectarlo correctamente
df_clean['track_id'] = df_clean['track_id'].replace(r'^\s*$', np.nan, regex=True)

# 4. Identificar y mostrar explícitamente las filas con valores nulos o incompletos
cols_criticas = ['track_id', 'track_name', 'track_artist', 'danceability', 'energy', 'valence', 'tempo', 'loudness']
nulos_detectados = df_clean[df_clean[cols_criticas].isnull().any(axis=1)]

print(f"⚠️ Se detectaron {len(nulos_detectados)} registros con valores nulos/incompletos:")
print(nulos_detectados[['track_id', 'track_name', 'track_artist', 'danceability', 'energy', 'valence', 'tempo', 'loudness']])

# 5. Eliminar las filas nulas detectadas
df_clean['track_id'] = df_clean['track_id'].replace(r'^\s*$', np.nan, regex=True)
df_clean = df_clean.dropna(subset=cols_criticas)

# 6. Resumen de distribución final
distribucion = df_clean['is_hit'].value_counts()

print("\n--- 📈 DISTRIBUCIÓN FINAL DEL DATASET ---")
print(f"Total Canciones Procesadas: {len(df_clean)}")
print(f"• Hits (is_hit = 1): {distribucion[1]} canciones ({distribucion[1]/len(df_clean):.1%})")
print(f"• No-Hits (is_hit = 0): {distribucion[0]} canciones ({distribucion[0]/len(df_clean):.1%})")

⚠️ Se detectaron 1 registros con valores nulos/incompletos:
                    track_id track_name track_artist  danceability  energy  \
1949  2teI76KKFE6qkpLZJs7tZ7    Make It     Berhanio           NaN     NaN   

      valence  tempo  loudness  
1949      NaN    NaN       NaN  

--- 📈 DISTRIBUCIÓN FINAL DEL DATASET ---
Total Canciones Procesadas: 4494
• Hits (is_hit = 1): 1227 canciones (27.3%)
• No-Hits (is_hit = 0): 3267 canciones (72.7%)


## 4. Exportación del Dataset Limpio
Guardamos la versión final procesada en `data/spotify_clean.csv` codificada en `utf-8` estándar para garantizar compatibilidad completa con Scikit-Learn y Streamlit.

In [78]:
# Crear carpeta si no existe
os.makedirs("../data", exist_ok=True)
OUTPUT_PATH = "../data/spotify_clean.csv"

# Guardar en CSV
df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Dataset procesado y exportado con éxito en: {OUTPUT_PATH}")

Dataset procesado y exportado con éxito en: ../data/spotify_clean.csv
